## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [19]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [ ]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        # 两层隐藏层 + 一层输出层
        self.d_in = 784
        self.d_hidden1 = 512
        self.d_hidden2 = 256
        self.d_out = 10

        he_init = tf.keras.initializers.HeNormal()
        out_init = tf.keras.initializers.GlorotUniform()

        # 第一层
        self.W1 = tf.Variable(he_init([self.d_in, self.d_hidden1]))
        self.b1 = tf.Variable(tf.zeros([self.d_hidden1]))

        # 第二层
        self.W2 = tf.Variable(he_init([self.d_hidden1, self.d_hidden2]))
        self.b2 = tf.Variable(tf.zeros([self.d_hidden2]))

        # 输出层
        self.W3 = tf.Variable(out_init([self.d_hidden2, self.d_out]))
        self.b3 = tf.Variable(tf.zeros([self.d_out]))

        self.vars = [
            self.W1, self.b1,
            self.W2, self.b2,
            self.W3, self.b3
        ]
        ####################

    def __call__(self, x, training=False):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        x = tf.reshape(tf.cast(x, tf.float32), [-1, self.d_in])

        # 标准化
        x = (x - 0.1307) / 0.3081

        # 第一层
        h1 = tf.matmul(x, self.W1) + self.b1
        h1 = tf.nn.relu(h1)

        if training:
            h1 = tf.nn.dropout(h1, rate=0.2)

        # 第二层
        h2 = tf.matmul(h1, self.W2) + self.b2
        h2 = tf.nn.relu(h2)

        if training:
            h2 = tf.nn.dropout(h2, rate=0.2)

        # 正确的输出层：h2 -> W3 -> logits
        logits = tf.matmul(h2, self.W3) + self.b3
        ####################
        return logits

model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [3]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    for g, v in zip(grads, trainable_vars):
        v.assign_sub(0.01*g)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [6]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 3.7155795 ; accuracy 0.12996666
epoch 1 : loss 3.3988006 ; accuracy 0.13275
epoch 2 : loss 3.1792188 ; accuracy 0.13483334
epoch 3 : loss 3.0136607 ; accuracy 0.13983333
epoch 4 : loss 2.883489 ; accuracy 0.14315
epoch 5 : loss 2.7778256 ; accuracy 0.14785
epoch 6 : loss 2.6894917 ; accuracy 0.15368333
epoch 7 : loss 2.6136234 ; accuracy 0.15968333
epoch 8 : loss 2.5469239 ; accuracy 0.16676667
epoch 9 : loss 2.4871745 ; accuracy 0.17426667
epoch 10 : loss 2.4328637 ; accuracy 0.1829
epoch 11 : loss 2.3829246 ; accuracy 0.19181667
epoch 12 : loss 2.3365746 ; accuracy 0.20146666
epoch 13 : loss 2.2932255 ; accuracy 0.21175
epoch 14 : loss 2.2524111 ; accuracy 0.22286667
epoch 15 : loss 2.2137578 ; accuracy 0.2343
epoch 16 : loss 2.1769633 ; accuracy 0.24506667
epoch 17 : loss 2.1417844 ; accuracy 0.2573
epoch 18 : loss 2.1080222 ; accuracy 0.26905
epoch 19 : loss 2.0755203 ; accuracy 0.28051665
epoch 20 : loss 2.044153 ; accuracy 0.29246667
epoch 21 : loss 2.0138192 ; acc